# ridhorezkyanwar-testing.ipynb
# Testing & Prediction Request ke Wine Quality API

Notebook ini digunakan untuk menguji sistem machine learning yang sudah di-deploy di Railway.


In [ ]:
import requests
import json
import pandas as pd

BASE_URL = 'https://wine-quality-mlops-production.up.railway.app/'

# Untuk testing lokal:
# BASE_URL = 'http://localhost:8080'

## Health Check

In [ ]:
response = requests.get(f'{BASE_URL}/health')
print(f'Status: {response.status_code}')
print(f'Response: {response.json()}')

Status: 200
Response: {'status': 'healthy'}


## Single Prediction - Good Wine

In [ ]:
# wine berkualitas tinggi (quality >= 6)
good_wine = {
    'fixed_acidity': 7.4,
    'volatile_acidity': 0.28,
    'citric_acid': 0.34,
    'residual_sugar': 1.2,
    'chlorides': 0.045,
    'free_sulfur_dioxide': 35.0,
    'total_sulfur_dioxide': 141.0,
    'density': 0.9940,
    'pH': 3.42,
    'sulphates': 0.68,
    'alcohol': 12.5
}

response = requests.post(f'{BASE_URL}/predict', json=good_wine)
print(f'Status: {response.status_code}')
result = response.json()
print(f'Probability: {result["probability"]}')
print(f'Label: {result["label"]}')

Status: 200
Probability: 0.9124
Label: good


## Single Prediction - Bad Wine

In [ ]:
# wine berkualitas rendah (quality < 6)
bad_wine = {
    'fixed_acidity': 11.2,
    'volatile_acidity': 0.92,
    'citric_acid': 0.01,
    'residual_sugar': 1.8,
    'chlorides': 0.075,
    'free_sulfur_dioxide': 17.0,
    'total_sulfur_dioxide': 60.0,
    'density': 0.9980,
    'pH': 3.16,
    'sulphates': 0.58,
    'alcohol': 9.8
}

response = requests.post(f'{BASE_URL}/predict', json=bad_wine)
print(f'Status: {response.status_code}')
result = response.json()
print(f'Probability: {result["probability"]}')
print(f'Label: {result["label"]}')

Status: 200
Probability: 0.28
Label: bad


## Batch Prediction dari Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('data/wine_quality.csv')
sample = df.sample(10, random_state=42)

results = []
for _, row in sample.iterrows():
    payload = {
        'fixed_acidity': float(row['fixed_acidity']),
        'volatile_acidity': float(row['volatile_acidity']),
        'citric_acid': float(row['citric_acid']),
        'residual_sugar': float(row['residual_sugar']),
        'chlorides': float(row['chlorides']),
        'free_sulfur_dioxide': float(row['free_sulfur_dioxide']),
        'total_sulfur_dioxide': float(row['total_sulfur_dioxide']),
        'density': float(row['density']),
        'pH': float(row['pH']),
        'sulphates': float(row['sulphates']),
        'alcohol': float(row['alcohol']),
    }
    resp = requests.post(f'{BASE_URL}/predict', json=payload)
    pred = resp.json()
    actual = 'good' if row['quality'] >= 6 else 'bad'
    results.append({
        'actual_quality': int(row['quality']),
        'actual_label': actual,
        'predicted_label': pred.get('label'),
        'probability': pred.get('probability'),
        'correct': actual == pred.get('label')
    })

results_df = pd.DataFrame(results)
print(results_df.to_string())
print(f'\nAkurasi pada 10 sampel: {results_df["correct"].mean():.2%}')

   actual_quality actual_label predicted_label  probability  correct
0               8         good            good       0.9392     True
1               5          bad             bad       0.2542     True
2               7         good            good       0.8878     True
3               6         good             bad       0.4327    False
4               6         good             bad       0.2695    False
5               6         good            good       0.7633     True
6               5          bad            good       0.6300    False
7               6         good            good       0.7234     True
8               5          bad             bad       0.2922     True
9               7         good            good       0.8999     True

Akurasi pada 10 sampel: 70.00%


## Cek Prometheus Metrics

In [ ]:
response = requests.get(f'{BASE_URL}/metrics')
print(f'Status: {response.status_code}')
for line in response.text.split('\n'):
    if 'prediction' in line and not line.startswith('#'):
        print(line)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Status: 200
prediction_requests_total{endpoint="/predict",method="POST",status="200"} 13.0
prediction_requests_created{endpoint="/predict",method="POST",status="200"} 1.7888479962332747e+09
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.005"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.01"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.025"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.05"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.075"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.1"} 11.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.25"} 12.0
prediction_request_latency_seconds_bucket{endpoint="/predict",le="0.5"} 13.0
prediction_request_latency_seconds_bucket{e